# **Exercicio Duelo de Modelos 4**

Nesta tarefa, vocês irão criar o seu próprio duelo de modelos, com o objetivo de superar os resultados apresentados em aula. O desafio é alcançar um desempenho superior ao que obtivemos, e para isso, será necessário aplicar todas as melhorias que vocês aprenderam ao longo dos módulos, utilizando a base de dados do Titanic.

**1. Escolha do Modelo:**
Selecione um dos modelos que foram explorados nos duelos de modelos ao longo do curso. Pode ser SVM, Random Forest, XGBoost, ou qualquer outro que tenhamos abordado.

**2. Aperfeiçoamento:**
**Aplique as técnicas que aprendemos para melhorar o desempenho do seu modelo:**

**Hiperparâmetros:** Utilize GridSearchCV ou RandomSearchCV para encontrar os melhores parâmetros.

**Cross Validation:** Avalie a robustez do modelo utilizando validação cruzada para garantir que ele generaliza bem.

**Balanceamento de Classes:** Se o seu modelo lida com problemas de classes desbalanceadas, explore técnicas como SMOTE, undersampling ou oversampling.

**Padronização e Normalização:** Lembre-se de padronizar os dados, especialmente se for usar modelos que são sensíveis à escala das variáveis.

**3. Submissão no Kaggle:**
Treine o seu modelo com os dados de treino e gere as previsões para os dados de teste. Lembre-se de que o conjunto de teste não possui a variável alvo (y_test), pois a avaliação será feita com base nas submissões no Kaggle.
Submeta suas previsões na competição do Titanic no Kaggle.

**4. Entrega:**
Envie o código que você desenvolveu, detalhando cada etapa do seu processo de modelagem, explicando as escolhas feitas e como essas ajudaram a melhorar o modelo.

Junto com o código, envie um print do seu score obtido na plataforma do Kaggle. Esse score será a sua métrica final de avaliação, mostrando como o seu modelo se compara com os demais.

**5. Competição Saudável:**
A ideia é trazer um senso de competição saudável, então não vale replicar exatamente o que fizemos na aula! Inove, explore novas combinações de parâmetros e técnicas, e mostre do que é capaz. O importante é exercitar o pensamento crítico e a capacidade de experimentar.

**Dicas Finais:**

Seja criativo e tenha um olhar crítico sobre o que pode ser melhorado.
Teste diferentes abordagens e não se prenda a um único caminho.
Lembre-se de que, mais do que alcançar o melhor score, o objetivo é aprender e aplicar o conhecimento de forma prática e eficaz.
Boa sorte! Estamos ansiosos para ver como cada um de vocês vai se sair nesse desafio e quais insights irão surgir dessa competição!

Ao final dessa atividade vocês terão participado da primeira competição publica de ciência de dados de vocês = )




## 1 - Importação das Bibliotecas e carregamento da base de dados

In [84]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [85]:
# Carregamento das bases de dados de treino e teste
train_df = pd.read_csv('train.csv')
teste_df = pd.read_csv('test.csv')

In [86]:
# Exibição da base treino com a variável target
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [87]:
# Exibição da base teste que possue somente as features
teste_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## 2 - Análise e tratamento dos dados

### A) Tipo de dados

In [88]:
# Análise dos tipos de dados da base de treino
train_df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [89]:
# Análise dos tipos de dados da base de teste
teste_df.dtypes

PassengerId      int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

As bases de dados de treino e teste apresentam estruturas e tipos de dados consistentes, divergindo exclusivamente pela presença da variável alvo (**target**) "`Survived`", contida apenas no conjunto de treino conforme as especificações do desafio.

A análise exploratória inicial identificou cinco variáveis qualitativas (do tipo **object**): `Name`, `Sex`, `Ticket`, `Cabin` e `Embarked`. As variáveis estritamente textuais — `Name` e `Ticket` — serão descartadas nesta etapa de modelagem. Por funcionarem essencialmente como identificadores de texto sem padrão estatístico direto, a permanência delas geraria ruído e não agregaria poder de generalização para os algoritmos.

Para o tratamento das variáveis categóricas remanescentes, foram adotadas abordagens distintas com base em suas características:
- `Sex`: Sendo uma variável binária (duas categorias), será transformada via Label Encoding, mapeando-a diretamente para valores 0 e 1.
- `Embarked`: Por apresentar três categorias distintas, será tratada com One-Hot Encoding (codificação binária dummy), evitando que o modelo presuma uma ordem hierárquica inexistente entre os portos de embarque.
- `Cabin`: Devido ao grande volume de dados nulos e alta fragmentação, aplicou-se uma técnica de engenharia de recursos para extrair a letra inicial de cada registro, que corresponde ao convés ocupado pelo passageiro. Posteriormente, essa nova estrutura será codificada via One-Hot Encoding, permitindo que o modelo capture o impacto da localização espacial no navio sobre a taxa de sobrevivência, sem perda drástica de representatividade dos dados.

In [90]:
# Exclusão das variáveis "Name e Ticket"
train_df = train_df.drop(['Name', 'Ticket'], axis=True)
teste_df = teste_df.drop(['Name', 'Ticket'], axis=True)

In [91]:
# Transformação da variável "Sex" com Label Encoder
# Chamando a função Label Encoder
label_encoder = LabelEncoder()

# Aplicação do Label Encoder nas coluas categóricas "Sex"
train_df['Sex'] = label_encoder.fit_transform(train_df['Sex'])
teste_df['Sex'] = label_encoder.fit_transform(teste_df['Sex'])

In [ ]:
# Verificação dos dados presentes na base de dados de treino e teste na variável "Cabin"
dados_cabine_train = train_df['Cabin'].unique()
dados_cabine_teste = teste_df['Cabin'].unique()

print(f'Dados presente na variável "Cabin" da base de dados treino:\n {dados_cabine_train}\n')
print(f'Dados presente na variável "Cabin" da base de dados teste:\n {dados_cabine_teste}')

Dados presente na variável "Cabin" da base de dados treino:
 [nan 'C85' 'C123' 'E46' 'G6' 'C103' 'D56' 'A6' 'C23 C25 C27' 'B78' 'D33'
 'B30' 'C52' 'B28' 'C83' 'F33' 'F G73' 'E31' 'A5' 'D10 D12' 'D26' 'C110'
 'B58 B60' 'E101' 'F E69' 'D47' 'B86' 'F2' 'C2' 'E33' 'B19' 'A7' 'C49'
 'F4' 'A32' 'B4' 'B80' 'A31' 'D36' 'D15' 'C93' 'C78' 'D35' 'C87' 'B77'
 'E67' 'B94' 'C125' 'C99' 'C118' 'D7' 'A19' 'B49' 'D' 'C22 C26' 'C106'
 'C65' 'E36' 'C54' 'B57 B59 B63 B66' 'C7' 'E34' 'C32' 'B18' 'C124' 'C91'
 'E40' 'T' 'C128' 'D37' 'B35' 'E50' 'C82' 'B96 B98' 'E10' 'E44' 'A34'
 'C104' 'C111' 'C92' 'E38' 'D21' 'E12' 'E63' 'A14' 'B37' 'C30' 'D20' 'B79'
 'E25' 'D46' 'B73' 'C95' 'B38' 'B39' 'B22' 'C86' 'C70' 'A16' 'C101' 'C68'
 'A10' 'E68' 'B41' 'A20' 'D19' 'D50' 'D9' 'A23' 'B50' 'A26' 'D48' 'E58'
 'C126' 'B71' 'B51 B53 B55' 'D49' 'B5' 'B20' 'F G63' 'C62 C64' 'E24' 'C90'
 'C45' 'E8' 'B101' 'D45' 'C46' 'D30' 'E121' 'D11' 'E77' 'F38' 'B3' 'D6'
 'B82 B84' 'D17' 'A36' 'B102' 'B69' 'E49' 'C47' 'D28' 'E17' 'A24' 'C5

In [92]:
# Extração da primeira letra da variável "Cabin" e subtituição de valores nulos por "U"
# Definição da extração das letras inicias das variáveis, assim como a substituição dos valor nulos por "U"
def extracao(variavel):
    var_teste = []
    for x in variavel:
        if str(x) == 'nan':
            var_teste.append('U')
        else:
            var_teste.append(str(x)[0])
    return var_teste

# Extração e substituição dos dados na base
train_df['Cabin'] =  extracao(train_df['Cabin'])
teste_df['Cabin'] = extracao(teste_df['Cabin'])

print(f'Exibição da susbtituição da base de treino: \n{train_df['Cabin']}\n')
print(f'Exibição da susbtituição da base de teste: \n{teste_df['Cabin']}')

Exibição da susbtituição da base de treino: 
0      U
1      C
2      U
3      C
4      U
      ..
886    U
887    B
888    U
889    C
890    U
Name: Cabin, Length: 891, dtype: object

Exibição da susbtituição da base de teste: 
0      U
1      U
2      U
3      U
4      U
      ..
413    U
414    C
415    U
416    U
417    U
Name: Cabin, Length: 418, dtype: object


In [93]:
# Transformação das variáveis "Embarked e Cabin" com One-Hot Encoding e exclusão das variáveis do tipo object
train_df = pd.get_dummies(train_df, columns=['Embarked', 'Cabin'], prefix=['Embarked', 'Cabin'])
teste_df = pd.get_dummies(teste_df, columns=['Embarked', 'Cabin'], prefix=['Embarked', 'Cabin'])


In [94]:
# Conferência dos tipos de dados da base de treino
train_df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Sex              int64
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Embarked_C        bool
Embarked_Q        bool
Embarked_S        bool
Cabin_A           bool
Cabin_B           bool
Cabin_C           bool
Cabin_D           bool
Cabin_E           bool
Cabin_F           bool
Cabin_G           bool
Cabin_T           bool
Cabin_U           bool
dtype: object

In [82]:
# Conferência dos tipos de dados da base de teste
teste_df.dtypes

PassengerId      int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Embarked_C        bool
Embarked_Q        bool
Embarked_S        bool
Cabin_A           bool
Cabin_B           bool
Cabin_C           bool
Cabin_D           bool
Cabin_E           bool
Cabin_F           bool
Cabin_G           bool
Cabin_U           bool
dtype: object

### B) Valores nulos

In [ ]:
# Verificação do total de valores nulos nas bases de treino
train_df.isnull().sum()

Base de teste:
PassengerId     0
Pclass          0
Sex             0
Age            86
SibSp           0
Parch           0
Fare            1
Embarked_C      0
Embarked_Q      0
Embarked_S      0
Cabin_A         0
Cabin_B         0
Cabin_C         0
Cabin_D         0
Cabin_E         0
Cabin_F         0
Cabin_G         0
Cabin_U         0
dtype: int64


In [123]:
# Verificação da porcentagem de valores nulos nas bases de treino
train_df.isnull().mean()

PassengerId    0.000000
Survived       0.000000
Pclass         0.000000
Sex            0.000000
Age            0.198653
SibSp          0.000000
Parch          0.000000
Fare           0.000000
Embarked_C     0.000000
Embarked_Q     0.000000
Embarked_S     0.000000
Cabin_A        0.000000
Cabin_B        0.000000
Cabin_C        0.000000
Cabin_D        0.000000
Cabin_E        0.000000
Cabin_F        0.000000
Cabin_G        0.000000
Cabin_T        0.000000
Cabin_U        0.000000
dtype: float64

In [119]:
# Verificação de valores nulos nas bases de teste
teste_df.isnull().sum()

PassengerId     0
Pclass          0
Sex             0
Age            86
SibSp           0
Parch           0
Fare            1
Embarked_C      0
Embarked_Q      0
Embarked_S      0
Cabin_A         0
Cabin_B         0
Cabin_C         0
Cabin_D         0
Cabin_E         0
Cabin_F         0
Cabin_G         0
Cabin_U         0
dtype: int64

In [124]:
# Verificação da porcentagem de valores nulos nas bases de teste
teste_df.isnull().mean()

PassengerId    0.000000
Pclass         0.000000
Sex            0.000000
Age            0.205742
SibSp          0.000000
Parch          0.000000
Fare           0.002392
Embarked_C     0.000000
Embarked_Q     0.000000
Embarked_S     0.000000
Cabin_A        0.000000
Cabin_B        0.000000
Cabin_C        0.000000
Cabin_D        0.000000
Cabin_E        0.000000
Cabin_F        0.000000
Cabin_G        0.000000
Cabin_U        0.000000
dtype: float64

O mapeamento de dados ausentes indicou que apenas as variáveis numéricas `Age` e `Fare` demandam tratamento, apresentando os seguintes índices de vacância:

- `Age`: Apresenta 19,8% de dados nulos na base de treino e 20,5% na base de teste.
- `Fare`:  Apresenta uma taxa residual de apenas 0,2% no conjunto de teste e 0% na base de treino. Por se tratar de um volume extremamente baixo em comparação ao total de registros, essa ausência é considerada estatisticamente irrelevante, permitindo uma correção segura.

Para corrigir essas omissões sem comprometer o volume amostral do dataset, será aplicado a estratégia de imputação pela média. Essa abordagem preenche as lacunas estatísticas de ambas as colunas, preservando a integridade estrutural das tabelas e garantindo a estabilidade necessária para a execução dos modelos.

In [125]:
# Preenchimento dos valores nulos das variáveis "Age e Fare" das bases de dados de treino e teste com a média
# Train
train_df['Age'].fillna(train_df['Age'].mean(), inplace=True)
train_df['Fare'].fillna(train_df['Fare'].mean(), inplace=True)

# Teste
teste_df['Age'].fillna(train_df['Age'].mean(), inplace=True)
teste_df['Fare'].fillna(teste_df['Fare'].mean(), inplace=True)

C:\Users\nneto\AppData\Local\Temp\ipykernel_18040\4260478402.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['Age'].fillna(train_df['Age'].mean(), inplace=True)
C:\Users\nneto\AppData\Local\Temp\ipykernel_18040\4260478402.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

In [126]:
# Confêrencia de valores nulos na bases de treino
train_df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked_C     0
Embarked_Q     0
Embarked_S     0
Cabin_A        0
Cabin_B        0
Cabin_C        0
Cabin_D        0
Cabin_E        0
Cabin_F        0
Cabin_G        0
Cabin_T        0
Cabin_U        0
dtype: int64

In [127]:
# Confêrencia de valores nulos na bases de teste
teste_df.isnull().sum()

PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked_C     0
Embarked_Q     0
Embarked_S     0
Cabin_A        0
Cabin_B        0
Cabin_C        0
Cabin_D        0
Cabin_E        0
Cabin_F        0
Cabin_G        0
Cabin_U        0
dtype: int64

### C) Separação das variáveis dependentes (Y) e independentes (X) no conjunto de treino